In [0]:
articles_schema = """article_id INTEGER, product_code INTEGER, prod_name STRING, product_type_no INTEGER, product_type_name	STRING,product_group_name STRING, graphical_appearance_no INTEGER, graphical_appearance_name STRING, colour_group_code INTEGER, colour_group_name STRING, perceived_colour_value_id INTEGER, perceived_colour_value_name STRING, perceived_colour_master_id INTEGER,	perceived_colour_master_name STRING, department_no INTEGER, department_name STRING, index_code STRING, index_name STRING, index_group_no INTEGER, index_group_name STRING, section_no INTEGER, section_name STRING, garment_group_no INTEGER, garment_group_name STRING, detail_desc STRING"""

In [0]:

from pyspark.sql.functions import col,current_timestamp

df_articles = (spark.readStream
      .format("cloudFiles")
      .option("cloudFiles.format","csv")
      .option("cloudFiles.schemaLocation","abfss://handm@dbprojectsstorage.dfs.core.windows.net/checkpoints/schema_infer/articles/")
      .option("cloudFiles.schemaEvolutionMode","rescue")
      .option("header",True)
      .schema(articles_schema)
      .load('abfss://handm@dbprojectsstorage.dfs.core.windows.net/landing/articles/'))

df_articles_metadata =( df_articles.withColumn("inget_time",current_timestamp())
                       .withColumn("file_path",col("_metadata.file_path")))

df_articles_writestream = (df_articles_metadata.writeStream
                           .format("delta")
                           .option("checkpointLocation","abfss://handm@dbprojectsstorage.dfs.core.windows.net/checkpoints/chkpoint/articles/")
                        #    .option("path","abfss://handm@dbprojectsstorage.dfs.core.windows.net/bronze/articles/")
                           .outputMode("append")
                           .trigger(availableNow=True)
                           .toTable("dev_handm.bronze.articles")
                           )
df_articles_writestream.awaitTermination()

In [0]:
%sql
SELECT * FROM csv.`abfss://handm@dbprojectsstorage.dfs.core.windows.net/landing/customers/`

In [0]:
customers_schema = "customer_id	STRING,FN STRING,Active STRING,club_member_status STRING, fashion_news_frequency STRING, age INTEGER, postal_code STRING"

In [0]:

from pyspark.sql.functions import col,current_timestamp

df_customers = (spark.readStream
      .format("cloudFiles")
      .option("cloudFiles.format","csv")
      .option("cloudFiles.schemaLocation","abfss://handm@dbprojectsstorage.dfs.core.windows.net/checkpoints/schema_infer/customers/")
      .option("cloudFiles.schemaEvolutionMode","rescue")
      .option("header",True)
      .schema(customers_schema)
      .load('abfss://handm@dbprojectsstorage.dfs.core.windows.net/landing/customers/'))

df_customers_metadata =( df_customers.withColumn("inget_time",current_timestamp())
                       .withColumn("file_path",col("_metadata.file_path")))

df_customers_writestream = (df_customers_metadata.writeStream
                           .format("delta")
                           .option("checkpointLocation","abfss://handm@dbprojectsstorage.dfs.core.windows.net/checkpoints/chkpoint/customers/")
                        #    .option("path","abfss://handm@dbprojectsstorage.dfs.core.windows.net/bronze/customers/")
                           .option("mergeSchema",True)
                           .outputMode("append")
                           .trigger(availableNow=True)
                           .toTable("dev_handm.bronze.customers")
                           )
df_customers_writestream.awaitTermination()

In [0]:
%sql
SELECT * FROM dev_handm.bronze.customers;

In [0]:
transactions_schema = "t_dat STRING,customer_id STRING,article_id INTEGER,price DOUBLE,sales_channel_id INTEGER"

In [0]:

from pyspark.sql.functions import col,current_timestamp

df_transactions = (spark.readStream
      .format("cloudFiles")
      .option("cloudFiles.format","csv")
      .option("cloudFiles.schemaLocation","abfss://handm@dbprojectsstorage.dfs.core.windows.net/checkpoints/schema_infer/transactions/")
      .option("cloudFiles.schemaEvolutionMode","rescue")
      .option("header",True)
      .schema(transactions_schema)
      .load('abfss://handm@dbprojectsstorage.dfs.core.windows.net/landing/transactions/'))

df_transactions_metadata =( df_transactions.withColumn("inget_time",current_timestamp())
                       .withColumn("file_path",col("_metadata.file_path")))

df_transactions_writestream = (df_transactions_metadata.writeStream
                           .format("delta")
                           .option("checkpointLocation","abfss://handm@dbprojectsstorage.dfs.core.windows.net/checkpoints/chkpoint/transactions/")
                           .option("mergeSchema",True)
                           .outputMode("append")
                           .trigger(availableNow=True)
                           .toTable("dev_handm.bronze.transactions")
                           )
df_transactions_writestream.awaitTermination()

In [0]:
%sql
DROP TABLE dev_handm.bronze.transactions;

In [0]:
%sql
SELECT * FROM dev_handm.bronze.transactions LIMIT 100

In [0]:
%sql
SELECT * FROM csv.`abfss://handm@dbprojectsstorage.dfs.core.windows.net/landing/transactions/`

In [0]:
dbutils.fs.mv("abfss://handm@dbprojectsstorage.dfs.core.windows.net/landing/transactions_train.csv","abfss://handm@dbprojectsstorage.dfs.core.windows.net/landing/transactions/")